# Day 55 · Exercise 5: Registration and Login

**What you'll build:** Implement `register_user(email, password, users)` and `login_user(email, password, users)`. These two functions are the complete auth handshake: register hashes and stores; login verifies and issues a JWT. Every auth system builds on exactly this pair.

## Setup (provided)

In [ ]:
import bcrypt as _bcrypt_lib
from jose import jwt, JWTError
from datetime import datetime, timedelta

SECRET_KEY  = "test-secret-for-exercise"
ALGORITHM   = "HS256"

def hash_password(password: str) -> str:
    return _bcrypt_lib.hashpw(password.encode(), _bcrypt_lib.gensalt()).decode()

def verify_password(plain: str, hashed: str) -> bool:
    return _bcrypt_lib.checkpw(plain.encode(), hashed.encode())

def create_token(data: dict, expires_in_minutes: int = 60) -> str:
    payload = data.copy()
    payload["exp"] = datetime.utcnow() + timedelta(minutes=expires_in_minutes)
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])


## Your Implementation

In [ ]:
_next_id = [1]  # mutable id counter

def register_user(email: str, password: str, users: dict) -> dict:
    """Register a new user, hashing their password.

    Args:
        email:    User's email address.
        password: Plaintext password (will be hashed before storing).
        users:    Mutable dict acting as the user store (email -> user record).
    Returns:
        Dict with {'id': int, 'email': str} of the new user.
    Raises:
        ValueError: If email is already registered.
    """
    # TODO: check if email in users → raise ValueError("Email already registered")
    # assign uid = _next_id[0]; _next_id[0] += 1
    # store users[email] = {"id": uid, "email": email, "hashed_password": hash_password(password)}
    # return {"id": uid, "email": email}
    raise NotImplementedError

def login_user(email: str, password: str, users: dict) -> str:
    """Authenticate a user and return a signed JWT.

    Args:
        email:    The user's email.
        password: Plaintext password to verify.
        users:    The user store dict.
    Returns:
        Signed JWT string.
    Raises:
        ValueError: If email not found or password is wrong.
    """
    # TODO: look up user = users.get(email)
    # if not user or not verify_password(password, user["hashed_password"]): raise ValueError
    # return create_token({"user_id": user["id"], "email": email})
    raise NotImplementedError


In [ ]:
_next_id = [1]

def register_user(email: str, password: str, users: dict) -> dict:
    if email in users:
        raise ValueError("Email already registered")
    uid = _next_id[0]
    _next_id[0] += 1
    users[email] = {"id": uid, "email": email, "hashed_password": hash_password(password)}
    return {"id": uid, "email": email}

def login_user(email: str, password: str, users: dict) -> str:
    user = users.get(email)
    if not user or not verify_password(password, user["hashed_password"]):
        raise ValueError("Invalid credentials")
    return create_token({"user_id": user["id"], "email": email})


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    store: dict = {}

    try:
        result = register_user("alice@example.com", "password123", store)
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: register_user not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, isinstance(result, dict) and "id" in result and result.get("email") == "alice@example.com",
         f"register returns {{id, email}} (got {result})")

    try:
        register_user("alice@example.com", "other", store)
        _chk(2, False, "duplicate email should raise ValueError")
    except ValueError:
        _chk(2, True, "duplicate email raises ValueError ✓")
    except Exception as e:
        _chk(2, False, f"raised {type(e).__name__} instead of ValueError")

    try:
        tok = login_user("alice@example.com", "password123", store)
    except NotImplementedError:
        for i in range(3, total + 1):
            print(f"  ❌ Check {i}: login_user not implemented")
        print(f"\nScore: {score} / {total}")
        return

    _chk(3, isinstance(tok, str) and tok.count(".") == 2,
         f"login_user returns a JWT string (got {str(tok)[:30]}...)")

    decoded = decode_token(tok)
    _chk(4, decoded.get("user_id") == result["id"],
         f"token user_id == {result['id']} (got {decoded.get('user_id')})")

    try:
        login_user("alice@example.com", "wrongpass", store)
        _chk(5, False, "wrong password should raise ValueError")
    except ValueError:
        _chk(5, True, "wrong password raises ValueError ✓")
    except Exception as e:
        _chk(5, False, f"raised {type(e).__name__} instead of ValueError")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Wire `register_user` and `login_user` into a mini FastAPI app: `POST /register` and `POST /login`. Add a protected `GET /me` using `Depends(get_current_user)` from Exercise 4. Test the full flow with TestClient — register → login → /me — without Uvicorn.

## Solution

<details>
<summary>Show solution</summary>

```python
_next_id = [1]

def register_user(email: str, password: str, users: dict) -> dict:
    if email in users:
        raise ValueError("Email already registered")
    uid = _next_id[0]
    _next_id[0] += 1
    users[email] = {"id": uid, "email": email, "hashed_password": hash_password(password)}
    return {"id": uid, "email": email}

def login_user(email: str, password: str, users: dict) -> str:
    user = users.get(email)
    if not user or not verify_password(password, user["hashed_password"]):
        raise ValueError("Invalid credentials")
    return create_token({"user_id": user["id"], "email": email})
```

**Why this works:** `register_user` rejects duplicates before storing (check-then-
act) and always hashes the password — never stores plaintext. `login_user` uses
`verify_password` (not `==`) to check the candidate against the stored hash, then
calls `create_token` to issue a JWT containing the user's id and email. The JWT is
what the client sends back on every protected request.

</details>